# 05 — Risk Target Creation

## AI-Based NIFTY 50 Portfolio Risk Prediction & Early Warning System

### Objective
Create the future-looking target variable that the ML model will predict.

Question:

> Given everything known about the portfolio today, does it experience a significant drawdown during the next 10 trading days?

We test **3%, 5%, and 10%** thresholds before choosing the primary event definition.


## 1. Leakage-safe time-series definition

At date `t`:

- **Features:** information known on or before `t`
- **Target:** portfolio behavior strictly after `t`

The future window is:

`t+1, ..., t+10`

The current day is excluded from the target calculation.

This is essential for avoiding look-ahead bias.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

INPUT_PATH = Path("../data/processed/portfolio_daily_baseline.csv")

portfolio = pd.read_csv(INPUT_PATH, parse_dates=["Date"])
portfolio = portfolio.sort_values("Date").reset_index(drop=True)

print("Rows:", len(portfolio))
print("Date range:", portfolio["Date"].min(), "to", portfolio["Date"].max())
print("Stock coverage:", portfolio["Number_of_Stocks"].min(),
      "to", portfolio["Number_of_Stocks"].max())


## 2. Coverage sensitivity

Compare minimum stock coverage rules of 30, 40, and 45 stocks.

This prevents us from silently treating the earliest, very sparse observations as a diversified portfolio.


In [ ]:
coverage_rows = []

for threshold in [30, 40, 45]:
    e = portfolio[portfolio["Number_of_Stocks"] >= threshold]
    coverage_rows.append({
        "Minimum_Stocks": threshold,
        "Eligible_Dates": len(e),
        "First_Eligible_Date": e["Date"].min(),
        "Last_Eligible_Date": e["Date"].max(),
        "Percent_of_All_Dates": 100 * len(e) / len(portfolio)
    })

coverage_sensitivity = pd.DataFrame(coverage_rows)
coverage_sensitivity


## 3. Primary coverage rule

We use:

> **At least 40 stocks must have a valid return on the prediction date.**

This removes the very sparse early history while retaining a large sample. The 30/45-stock results remain as sensitivity analysis.


In [ ]:
MIN_STOCKS = 40

eligible = portfolio[
    portfolio["Number_of_Stocks"] >= MIN_STOCKS
].copy().reset_index(drop=True)

print("Eligible dates:", len(eligible))
print("First eligible date:", eligible["Date"].min())
print("Last eligible date:", eligible["Date"].max())


## 4. Define future 10-trading-day drawdown

For every eligible date `t`:

`Future_Drawdown_10D = min(V[t+1:t+10]) / V[t] - 1`

Plain English:

> How far below today's portfolio value does the portfolio fall at its worst point during the next 10 trading days?


In [ ]:
values = eligible["Portfolio_Value"].to_numpy()
horizon = 10

future_dd = np.full(len(eligible), np.nan)
future_min = np.full(len(eligible), np.nan)
future_end = np.full(len(eligible), np.nan)
future_end_date = pd.Series(pd.NaT, index=eligible.index, dtype="datetime64[ns]")

for i in range(len(eligible) - horizon):
    future_values = values[i+1:i+horizon+1]
    future_min[i] = future_values.min()
    future_dd[i] = future_min[i] / values[i] - 1
    future_end[i] = values[i+horizon]
    future_end_date.iloc[i] = eligible.loc[i+horizon, "Date"]

eligible["Future_Min_Value_10D"] = future_min
eligible["Future_End_Value_10D"] = future_end
eligible["Future_Drawdown_10D"] = future_dd
eligible["Future_Window_End"] = future_end_date.values

eligible[[
    "Date", "Portfolio_Value",
    "Future_Drawdown_10D", "Future_Window_End"
]].head(15)


## 5. Candidate event labels

A target is positive when future drawdown reaches or exceeds the threshold:

- `Future_Drawdown_10D <= -0.03` → 3% event
- `Future_Drawdown_10D <= -0.05` → 5% event
- `Future_Drawdown_10D <= -0.10` → 10% event

The final 10 dates have no complete future window, so their labels stay missing.


In [ ]:
for pct in [3, 5, 10]:
    col = f"Target_DD_{pct}Pct_10D"

    eligible[col] = np.where(
        eligible["Future_Drawdown_10D"].notna(),
        (eligible["Future_Drawdown_10D"] <= -pct/100).astype(float),
        np.nan
    )


## 6. Compare class balance

The target should be financially meaningful and statistically learnable.

We therefore compare positive-event frequency across all three thresholds.


In [ ]:
target_rows = []

for pct in [3, 5, 10]:
    col = f"Target_DD_{pct}Pct_10D"
    known = eligible[col].dropna()

    positives = int((known == 1).sum())
    negatives = int((known == 0).sum())

    target_rows.append({
        "Target": col,
        "Threshold": f"{pct}%",
        "Horizon_Trading_Days": 10,
        "Known_Observations": len(known),
        "Positive_Events": positives,
        "Negative_Events": negatives,
        "Positive_Rate_Percent": 100 * positives / len(known)
    })

target_distribution = pd.DataFrame(target_rows)
target_distribution


## 7. Inspect the future-drawdown distribution

The distribution helps us understand whether 3%, 5%, and 10% represent common, moderate, or rare events.


In [ ]:
known_future_dd = eligible["Future_Drawdown_10D"].dropna()

print(
    known_future_dd.describe(
        percentiles=[.01,.05,.10,.25,.50,.75,.90,.95,.99]
    )
)


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(known_future_dd, bins=60)
plt.axvline(-0.03, linestyle="--", label="3%")
plt.axvline(-0.05, linestyle="--", label="5%")
plt.axvline(-0.10, linestyle="--", label="10%")
plt.title("Worst Future 10-Day Portfolio Drawdown")
plt.xlabel("Future Drawdown")
plt.ylabel("Number of Dates")
plt.legend()
plt.tight_layout()
plt.show()


## 8. Audit actual target events

Inspect real dates where the 5% event occurs. This is a sanity check on the label-generation logic.


In [ ]:
audit_cols = [
    "Date", "Portfolio_Value", "Current_Drawdown",
    "Future_Drawdown_10D", "Future_Window_End",
    "Target_DD_3Pct_10D", "Target_DD_5Pct_10D", "Target_DD_10Pct_10D"
]

eligible.loc[
    eligible["Target_DD_5Pct_10D"] == 1,
    audit_cols
].head(20)


## 9. Overlapping future windows

Adjacent prediction dates can share future observations.

That is expected for a rolling early-warning system.

### Consequence

Later modeling must use **chronological time-based splits**, not random train/test splitting.


## 10. Leakage checklist

### Allowed features
- today's volatility
- today's drawdown
- today's momentum
- today's concentration
- today's sector exposure

### Forbidden features
- future returns
- future volatility
- future drawdown
- future prices
- any statistic using observations after `t`

`Future_Drawdown_10D` is a **label-generation field**, not a model feature.


## 11. Save outputs

Main dataset:

`data/processed/portfolio_risk_targets.csv`

Reports:

- `reports/portfolio_coverage_sensitivity.csv`
- `reports/risk_target_distribution.csv`
- `reports/risk_target_audit_sample.csv`


In [ ]:
eligible.to_csv(
    "../data/processed/portfolio_risk_targets.csv",
    index=False
)

coverage_sensitivity.to_csv(
    "../reports/portfolio_coverage_sensitivity.csv",
    index=False
)

target_distribution.to_csv(
    "../reports/risk_target_distribution.csv",
    index=False
)

eligible[audit_cols].dropna(
    subset=["Future_Drawdown_10D"]
).head(30).to_csv(
    "../reports/risk_target_audit_sample.csv",
    index=False
)

print("Step 05 outputs saved.")


# Final methodology

### Portfolio eligibility
At least **40 stocks** with valid same-day returns.

### Prediction horizon
**10 trading days ahead.**

### Candidate targets
**3%, 5%, and 10% future drawdown.**

We select the primary threshold after reviewing the actual event frequency.

### Next notebook

`06_portfolio_feature_engineering.ipynb`

will combine stock-level features with portfolio-level risk and concentration signals.

Core rule:

> **Features use information available by date t; the target describes what happens after date t.**
